[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/06_phase3_stage5_realtime_amortised.ipynb)

# 06 · Stage 5: amortised, real-time risk (headline)

**Phase:** Phase 3: real-time

**Goal:** One network over a box of market parameters θ = (vol level s, jump intensity λ, mean jump μ_J) at d = 2 and 5: accuracy across θ, latency per VaR/ES query vs Monte Carlo at matched tail error, break-even query count, Euler risk contributions and what-if curves (RQ5).

**Outputs:** `figures/stage5_amortised_d<d>/fig01–fig05`, `results/stage5_amortised_d<d>/`

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

In [ ]:
# @title Setup: repo on Google Drive, install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # keep figures/results across Colab sessions (recommended)
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
else:
    !git -C "{ROOT}" pull -q --ff-only
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# autoreload picks up library edits without a restart. Colab's IPython build imports the
# removed `imp` module on Python >= 3.12, so it is optional: if it is skipped, use
# Runtime > Restart session after editing pinn_tailrisk/.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload unavailable here; restart the session after editing the library")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

## 1. Method in one paragraph
Stage 4 trained one network per market. Here θ = (s, λ, μ_J) are extra network inputs (s scales every vol;
λ is sampled log-uniformly; correlations, loadings and σ_J stay fixed), and every collocation, initial-condition
and normalisation point gets its own θ drawn from the training box. The PIDE coefficients, the exact initial
density at t0 and the importance proposal are all evaluated at each point's θ. After one training run, a **new
market state is one forward pass**: draw from the proposal at θ, weight by the network density, sort.
Any portfolio w reuses the same draws.

Scope: d ≤ 5 (Stage 4 found d = 10 is where the method breaks down). Truth for every (θ, w, horizon) is the
closed-form portfolio law. MC runs on the same GPU, so latency is like-for-like.

In [ ]:
print(open('configs/stage5_amortised.yaml').read())

## 2. Train and evaluate (d = 2, 5)
Resumable: finished runs are skipped. Estimated (not yet measured) **about 1 h on a T4** for both, including
evaluation. A crash after training can be re-evaluated without retraining: add `--eval-only`.

In [ ]:
import pathlib
STEPS = 30000
for d in (2, 5):
    run = f"stage5_amortised_d{d}"
    if pathlib.Path(f"results/{run}/metrics.json").exists():
        print("skip", run); continue
    !python scripts/stage5_amortised.py --d {d} --steps {STEPS}

# d = 10: not pursued (Stage 4 breakdown point). Skeleton for future work:
# !python scripts/stage5_amortised.py --d 10 --steps 60000

## 3. Figures and what to check

| Figure | Pass criterion |
|---|---|
| fig01_training | all loss terms fall; tracked ES₉₉ error at the base market drops to about 1% |
| fig02_accuracy_theta | median ES₉₉ error ≲ 1–2% over most of the (s, λ) box, at the level of MC with the same N; edges of the box may be worse |
| fig03_latency | error-vs-latency curves on the same GPU; the title states the speed-up at the PINN's error level |
| fig04_breakeven | queries after which training time is paid back (or "no break-even" if MC is faster per query) |
| fig05_euler_whatif | PINN Euler contributions match the exact bars; PINN what-if curves track the exact ES smoothly across λ and s |

In [ ]:
for d in (2, 5):
    print(f"======== d = {d}")
    show_figures(f"stage5_amortised_d{d}")

## 4. Numbers

In [ ]:
rows = []
for d in (2, 5):
    m = load_metrics(f"stage5_amortised_d{d}")
    for h in ("1d", "10d"):
        g, b = m["es99_median_over_grid"][h], m["breakeven"][h]
        rows.append(dict(d=d, horizon=h, es99_pinn=g["es_rel_pinn"], es99_mc=g["es_rel_mc"], es99_is_floor=g["es_rel_is_exact"],
                         pinn_ms=b["pinn_ms"], mc_ms_matched=b["mc_ms"], speedup=b["speedup"],
                         breakeven_queries=b["breakeven_queries"], train_min=m["train_time_s"] / 60))
display(pd.DataFrame(rows).style.format(precision=4))
for d in (2, 5):
    print(d, "worst theta:", load_metrics(f"stage5_amortised_d{d}")["es99_worst_theta"])